In [ ]:
# PREÁMBULO: carga de datos del notebook de preparación


import pandas as pd
import os

# Ruta relativa al CSV generado por el otro notebook
ruta_csv = os.path.join("..", "data", "close_approaches.csv")

if not os.path.exists(ruta_csv):
    raise FileNotFoundError(
        f"No se encontró el CSV en {ruta_csv}. "
        f"Ejecutar primero data/ProyectoNeoRework_data.ipynb")

df = pd.read_csv(ruta_csv)

# Limpieza de fechas (quitar incertidumbre ±)
def limpiar_fecha(fecha_str):
    if isinstance(fecha_str, str):
        return fecha_str.split("±")[0].strip()
    return fecha_str

df["Close-Approach (CA) Date"] = df["Close-Approach (CA) Date"].apply(limpiar_fecha)

print(f"Dataset cargado: {len(df):,} registros")

## Limpieza de datos

In [ ]:
df.isna().sum()

In [ ]:
df.dropna()

# Preparacion de modelos

Ahora se implementarán modelos de regresión junto con técnicas de machine learning, específicamente PCA y K-Means, con el objetivo de realizar un análisis exploratorio avanzado del espacio de datos. Esto permitirá evaluar la estructura subyacente, identificar patrones y medir la efectividad de estos enfoques, determinando si el problema requiere la aplicación de métodos más sofisticados o de mayor capacidad predictiva.


## PCA

El PCA (Análisis de Componentes Principales) es una técnica de reducción de dimensionalidad que transforma un conjunto de variables originales, posiblemente correlacionadas, en un nuevo conjunto de variables no correlacionadas llamadas componentes principales. Su funcionamiento se basa en identificar las direcciones del espacio de datos donde la varianza es máxima, utilizando la descomposición en autovalores y autovectores de la matriz de covarianza. De esta forma, los primeros componentes concentran la mayor cantidad de información, permitiendo proyectar los datos en un espacio de menor dimensión con mínima pérdida de variabilidad, lo que facilita su análisis, visualización y modelado.


### Librerias

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

### Preparacion de datos




In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
random_state = 20

# Variables que vamos a usar para el análisis
features = ["CA DistanceNominal (au)", "V relative(km/s)",
            "V infinity(km/s)", "H(mag)", "Diameter(km)"]

# Eliminamos filas con valores faltantes en estas features
X = df[features].dropna()

# Estandarización: cada variable pasa a tener media 0 y desv. estándar 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Dataset final: {X_scaled.shape[0]} muestras x {X_scaled.shape[1]} features")

In [ ]:
# PCA completo: extraemos las 5 componentes principales
# Esto nos permite ver cuánta varianza captura cada una
pca_full = PCA(random_state=random_state)
coords_full = pca_full.fit_transform(X_scaled)

# Varianza explicada por cada componente
exp_var = pca_full.explained_variance_ratio_ * 100
var_acum = np.cumsum(exp_var)

print("Varianza explicada por componente:")
for i, var in enumerate(exp_var):
    print(f"  PC{i+1}: {var:.1f}%")
print(f"\nAcumulado PC1+PC2: {var_acum[1]:.1f}%")

# Loadings: qué variables pesan más en cada componente
# Nos ayuda a interpretar el significado físico de cada PC
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f"PC{i+1}" for i in range(5)],
    index=features
)
print("\nLoadings (contribución de cada variable):")
print(loadings.round(3))

# Interpretación rápida: top 2 variables por componente
for pc in ["PC1", "PC2"]:
    top = loadings[pc].abs().sort_values(ascending=False).head(2)
    print(f"\n{pc} -> dominado por: {', '.join(top.index)}")

# Decisión: ¿PCA es suficiente o necesitamos clustering?
PCA_UMBRAL = 80.0
pca_suficiente = var_acum[1] >= PCA_UMBRAL
print(f"\n{'✓' if pca_suficiente else '✗'} PCA {'suficiente' if pca_suficiente else 'insuficiente'} "
      f"(PC1+PC2={var_acum[1]:.1f}% {'≥' if pca_suficiente else '<'} {PCA_UMBRAL:.0f}%)")

# Proyección a 2D para visualización
coords_2d = coords_full[:, :2]

### Visualizacion

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(coords_2d[:, 0], coords_2d[:, 1],
            c=X["Diameter(km)"], cmap="plasma", s=2, alpha=0.3)
plt.colorbar(label="Diameter (km)")
plt.title("NEOs en espacio PCA")
plt.xlabel(f"PC1 ({exp_var[0]:.1f}% varianza)")
plt.ylabel(f"PC2 ({exp_var[1]:.1f}% varianza)")
plt.show()

if pca_suficiente:
    print("PCA captura suficiente varianza. Los loadings revelan la estructura fisica.")
else:
    print("PCA insuficiente. Se aplicara K-Means sobre las 5 dimensiones originales.")

### Nota

PC1 y PC2 son ejes artificiales creados por el PCA que resumen la información de las 5 variables originales en 2 dimensiones. La interpretación física de cada PC se obtiene de los loadings (pesos de cada variable original en cada componente). Los valores en estos ejes no tienen unidades físicas directas sino que representan qué tan lejos está cada asteroide del promedio en esas direcciones. Si PC1+PC2 >= 80%, el PCA es suficiente para entender la estructura de los datos; de lo contrario, se requiere K-Means.

### Interpretacion

El PCA completo revela cuánta varianza explica cada componente y qué variables físicas dominan cada eje (loadings). Si PC1+PC2 >= 80%, la proyección 2D retiene suficiente información y no es necesario clusterizar. Si es menor, se procede con K-Means sobre los 5D escalados para capturar la estructura que el PCA pierde.

## K-Means

K-Means es un algoritmo de aprendizaje no supervisado utilizado para agrupar datos en K clusters según su similitud. Funciona asignando cada punto al centroide más cercano y recalculando iterativamente estos centroides como el promedio de los puntos del grupo, con el objetivo de minimizar la variabilidad interna de cada cluster (distancia entre los puntos y su centro). De esta manera, permite identificar patrones o estructuras en los datos sin necesidad de una variable objetivo.

### Librerias

In [ ]:
from sklearn.cluster import KMeans

### Metodo del codo

El método del codo se utiliza para determinar el número óptimo de clusters (K) en algoritmos como K-Means. Consiste en ejecutar el modelo con distintos valores de K y medir la suma de errores dentro de los clusters (inercia o WCSS). A medida que K aumenta, el error disminuye, pero llega un punto donde la mejora deja de ser significativa; ese punto de inflexión, que visualmente parece un “codo” en la gráfica, indica el número adecuado de clusters, ya que representa un equilibrio entre simplicidad del modelo y buena representación de los datos.

In [ ]:
# Buscamos el k óptimo probando de 2 a 10 clusters
k_range = range(2, 11)
inercias, sil_scores = [], []

for k in k_range:
    # Entrenamos K-Means para cada k y medimos calidad
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    etiq = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, etiq))

# Dos criterios para elegir k:
# - Codo: donde la inercia deja de bajar rápido
# - Silhouette: qué tan bien separados están los clusters
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(k_range, inercias, marker="o")
ax1.set_title("Método del codo")
ax1.set_xlabel("k")
ax1.set_ylabel("Inercia")
ax1.set_xticks(list(k_range))
ax1.grid(True, lw=0.5)

ax2.plot(k_range, sil_scores, marker="s", color="green")
ax2.set_title("Silhouette Score")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette")
ax2.set_xticks(list(k_range))
ax2.grid(True, lw=0.5)

plt.tight_layout()
plt.show()

# Elegimos el k con mejor silhouette (más separación entre clusters)
k_opt = k_range[np.argmax(sil_scores)]
print(f"k óptimo por silhouette: {k_opt} (silhouette={max(sil_scores):.4f})")

### Aplicacion del modelo (5D escalados, no sobre PCA)


In [ ]:
# Clustering final en el espacio completo (5 dimensiones)
# Es importante hacerlo aquí, no en el espacio reducido de PCA
km = KMeans(n_clusters=k_opt, random_state=random_state, n_init=10)
etiquetas = km.fit_predict(X_scaled)

# Proyectamos a 2D solo para visualizar, no para clusterizar
coords_2d = PCA(n_components=2, random_state=random_state).fit_transform(X_scaled)

print(f"Clusters encontrados: k={k_opt}")
print(f"Silhouette del modelo final: {silhouette_score(X_scaled, etiquetas):.4f}")

### Visualizacion

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(coords_2d[:, 0], coords_2d[:, 1],
            c=etiquetas, cmap="tab10", s=2, alpha=0.3)
plt.title(f"Clusters K-Means k={k_opt} (proyectados en PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### Distribucion de clusters

In [ ]:
# Agregamos las etiquetas de cluster al dataset original
X_con_cluster = X.copy()
X_con_cluster["Cluster"] = etiquetas

# Caracterizamos cada cluster con las medias de sus variables
print("Medias por cluster (escala original):")
print(X_con_cluster.groupby("Cluster")[features].mean().round(4))

# Interpretación: comparar estos valores nos dice qué tipo de NEOs agrupa cada cluster

### Interpretaciones
El análisis de K-Means sobre las 5 dimensiones originales (no sobre PCA) revela la estructura real de los datos. El k óptimo se elige maximizando el silhouette score, que mide cohesión intra-cluster vs. separación inter-cluster. La tabla de medias por cluster caracteriza cada grupo en las variables físicas originales, permitiendo interpretar qué tipo de NEO agrupa cada uno.

## t-SNE (t-distributed Stochastic Neighbor Embedding)

Técnica de reducción de dimensionalidad no lineal, diseñada principalmente para visualizar datos de alta dimensión en 2D o 3D.

Su principio central es preservar las relaciones locales: si dos puntos son cercanos en el espacio original, deben seguir siéndolo en la proyección. Para lograrlo:

1) Modela la similitud entre puntos del espacio original como probabilidades.
2) Hace lo mismo en el espacio reducido, usando una distribución t de Student (de ahí la "t").
3) Ajusta iterativamente la proyección para que ambas distribuciones sean lo más parecidas posible.

In [ ]:
import numpy as np
import pandas as pd
import optuna
from openTSNE import TSNE
from sklearn.manifold import trustworthiness

In [ ]:
# Suprimir logs de Optuna para mantener la salida limpia
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Número de combinaciones de hiperparámetros a explorar
N_TRIALS = 200

def objective(trial):
    """Evalúa una configuración de t-SNE y devuelve su trustworthiness."""

    # Espacio de búsqueda de hiperparámetros
    params = {
        "perplexity":         trial.suggest_int("perplexity", 2, 100),
        "early_exaggeration": trial.suggest_int("early_exaggeration", 4, 50),
        "learning_rate":      trial.suggest_int("learning_rate", 100, 1000),
        "n_iter":             trial.suggest_int("n_iter", 250, 2000),
    }

    # Ajuste inicial del embedding
    tsne = TSNE(**params, random_state=random_state, verbose=False)
    embedding = tsne.fit(X_scaled)

    # Refinamiento adicional del embedding
    embedding.optimize(n_iter=1500, learning_rate=500, inplace=True, verbose=False)

    # Métricas puramente geométricas, sin labels
    trust = trustworthiness(X_scaled, embedding, n_neighbors=15)
    kl    = float(embedding.kl_divergence)  # openTSNE lo guarda en el objeto

    # Guardar métricas como atributos del trial para análisis posterior
    trial.set_user_attr("trustworthiness", round(trust, 4))
    trial.set_user_attr("kl_divergence",   round(kl, 4))

    return trust  # maximizar preservación de vecindarios


# Configuración del estudio: maximizar trustworthiness con muestreo bayesiano
study = optuna.create_study(
    direction  = "maximize",
    study_name = "tsne_optimization",
    sampler    = optuna.samplers.TPESampler(seed=42),
    pruner     = optuna.pruners.MedianPruner(),
)

# Ejecución paralela de los trials
study.optimize(
    objective,
    n_trials          = N_TRIALS,
    n_jobs            = -1,
    show_progress_bar = True,
)

print("\n Mejores parámetros encontrados:")
print(study.best_params)
print(f"   Trustworthiness: {study.best_value:.4f}")

# Recolección de resultados de todos los trials
rows = []
for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        rows.append({
            **trial.params,
            "trustworthiness": trial.user_attrs.get("trustworthiness"),
            "kl_divergence":   trial.user_attrs.get("kl_divergence"),
            "status": "ok",
        })
    else:
        rows.append({**trial.params, "status": "error/pruned"})

# Exportar resultados a CSV
df_results = pd.DataFrame(rows)
df_results.to_csv("tsne_optuna_results.csv", index=False)
print("✓ Guardado en tsne_optuna_results.csv")

# Top 10 configuraciones por trustworthiness
print("\n Top 10 configuraciones:")
print(
    df_results[df_results["status"] == "ok"]
    .sort_values("trustworthiness", ascending=False)
    .head(10)
    .to_string(index=False)
)

In [ ]:
TOP_N = 10

df_results = pd.read_csv("tsne_optuna_results.csv")
top_configs = (
    df_results[df_results["status"] == "ok"]
    .sort_values("trustworthiness", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

for rank, row in top_configs.iterrows():
    params = {
        "perplexity":         int(row["perplexity"]),
        "early_exaggeration": int(row["early_exaggeration"]),
        "learning_rate":      int(row["learning_rate"]),
        "n_iter":             int(row["n_iter"]),
    }

    tsne      = TSNE(**params, random_state=random_state, verbose=False)
    embedding = tsne.fit(X_scaled)
    embedding.optimize(n_iter=1500, learning_rate=500, inplace=True, verbose=False)

    df_emb = pd.DataFrame({
        "x": embedding[:, 0],
        "y": embedding[:, 1],
        # métricas de la config
        "trustworthiness": row["trustworthiness"],
        "kl_divergence":   row["kl_divergence"],
        # parámetros usados
        "perplexity":         params["perplexity"],
        "early_exaggeration": params["early_exaggeration"],
        "learning_rate":      params["learning_rate"],
        "n_iter":             params["n_iter"],
    })

    filename = (
        f"tsne_rank{rank+1:02d}"
        f"_trust{row['trustworthiness']:.4f}"
        f"_kl{row['kl_divergence']:.4f}"
        f"_perp{params['perplexity']}"
        f"_ee{params['early_exaggeration']}"
        f"_lr{params['learning_rate']}"
        f"_iter{params['n_iter']}"
        f".csv"
    )
    df_emb.to_csv(filename, index=False)
    print(f"[{rank+1:>2}/{TOP_N}] ✓ {filename}")

print(f"\nListos {TOP_N} CSVs con coordenadas x/y puras del embedding.")